In [ ]:
# 读取文件夹结构
import os
# 指定路径
root_path = r"/Users/zhangnan/日常文件/ai/每周下载和预测/2 20250908/20250908-tp/20250908-tp 数据"
# 遍历文件夹结构
for root, dirs, files in os.walk(root_path):    
    print(f"当前文件夹: {root}")        
    # 判断是否有文件    
    if files:        
        display_files = files[:8]        
        print(f"  ✅ 文件数量: {len(files)}，展示前八个文件:")        
        for file in display_files:            
            print(f"    📄 {file}")    
        else:        
            print("  ⚠️ 该文件夹中没有文件")        
            if dirs:            
                print(f"  📁 包含的子文件夹: {', '.join(dirs)}")        
            else:            
                print("  ❌ 也没有子文件夹")    
        print()  # 空行用于分隔输出


In [ ]:
# 读取nc文件信息
import xarray as xr
# 定义文件路径
file_path = '/Users/zhangnan/日常文件/ai/每周下载和预测/2 20250908/20250908-tp/20250908-tp 数据/ERA5-daily-800hPa-CloudFraction-20250908.nc'
# 打印原文件路径
print(f"文件路径: {file_path}")
# 打开NetCDF文件
ds = xr.open_dataset(file_path)
# 输出数据集的基本信息
print(ds)
# 如果需要查看数据集的变量列表，可以使用
#print(ds.variables)
# 如果需要查看数据集的维度，可以使用
#print(ds.dims)


In [ ]:
# 1 处理月最新数据
import xarray as xr
import numpy as np
import os

# === 路径设置 ===
azn_file = "/Users/zhangnan/日常文件/ai/每周下载和预测/53 20260831/数据/download/ERA5-monthly-single level-AZN-20260831.nc"
tp_file = "/Users/zhangnan/日常文件/ai/每周下载和预测/53 20260831/数据/download/ERA5-monthly-single level-tp-20260831.nc"
output_path = "/Users/zhangnan/日常文件/ai/每周下载和预测/53 20260831/20260831-tp/20260831-tp-1/tp_merged_1.5deg_no_norm_20260831.nc"

os.makedirs(os.path.dirname(output_path), exist_ok=True)

# === 读取 AZN 文件 (包含 u10, v10, sst, t2m, msl) ===
print(f"📥 读取 AZN 文件: {azn_file}")
azn_ds = xr.open_dataset(azn_file)

# 子采样到 1.5° (每6个点取一个)
azn_sub = azn_ds.isel(latitude=slice(None, None, 6), longitude=slice(None, None, 6))

# 删除 expver 防止合并出错
if "expver" in azn_sub.coords:
    azn_sub = azn_sub.drop_vars("expver")

# 填充 NaN
for varname in azn_sub.data_vars:
    values = azn_sub[varname].values
    mask = np.isnan(values)
    if np.any(mask):
        mean_val = np.nanmean(values)
        values[mask] = mean_val
        azn_sub[varname].values = values

# === 读取降水文件 tp ===
print(f"📥 读取 TP 文件: {tp_file}")
tp_ds = xr.open_dataset(tp_file)
tp_sub = tp_ds["tp"].isel(latitude=slice(None, None, 6), longitude=slice(None, None, 6))

# 空间插值到 AZN 子采样网格
tp_sub = tp_sub.interp_like(azn_sub["u10"])

# 填充 NaN
values = tp_sub.values
mask = np.isnan(values)
if np.any(mask):
    mean_val = np.nanmean(values)
    values[mask] = mean_val
    tp_sub.values = values

# 删除 expver
if "expver" in tp_sub.coords:
    tp_sub = tp_sub.drop_vars("expver")

# === 合并所有变量 ===
merged_ds = azn_sub
merged_ds["tp"] = tp_sub

# === 时间维重命名为 time ===
if "valid_time" in merged_ds.dims or "valid_time" in merged_ds.coords:
    merged_ds = merged_ds.rename({"valid_time": "time"})

# === 排序并去重时间索引 ===
merged_ds = merged_ds.sortby("time")
merged_ds = merged_ds.sel(time=~merged_ds.get_index("time").duplicated())

# === 保存结果 ===
merged_ds.to_netcdf(output_path)
print(f"✅ 数据已保存: {output_path}")

In [ ]:
# 2 推理2026年01月的数据
import xarray as xr
import numpy as np
import torch
import torch.nn as nn
from tqdm import tqdm
import pandas as pd
import os

# ===== 参数设置 =====
timesteps = 15
variables = ["u10", "v10", "sst", "t2m", "msl", "tp"]
target_var = "tp"
target_idx = variables.index(target_var)
data_path = "/Users/zhangnan/日常文件/ai/每周下载和预测/53 20260831/20260831-tp/20260831-tp-1/tp_merged_1.5deg_no_norm_20260831.nc"
model_path = "/Users/zhangnan/日常文件/ai/每周下载和预测/推理用/AZN/月模型/PR/climate_tcn_with_timeencoding_150_1month_tp.pth"
output_path = "/Users/zhangnan/日常文件/ai/每周下载和预测/53 20260831/20260831-tp/20260831-tp-1/预测结果-tp-2026年8月.nc"
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# ===== 模型结构（和训练时一致）=====
class ChEncoder(nn.Module):
    def __init__(self, in_channels, embed_dim):
        super().__init__()
        self.spatial_encoder = nn.Sequential(
            nn.Conv2d(in_channels, 16, 3, padding=1), nn.ReLU(),
            nn.Conv2d(16, 32, 3, padding=1), nn.ReLU(),
            nn.AdaptiveAvgPool2d((4, 4))
        )
        self.fc = nn.Linear(32 * 4 * 4, embed_dim)

    def forward(self, x):
        x = self.spatial_encoder(x)
        x = x.flatten(1)
        return self.fc(x)

class TemporalConvNet(nn.Module):
    def __init__(self, input_size, num_channels, kernel_size=2, dropout=0.2):
        super().__init__()
        layers = []
        for i in range(len(num_channels)):
            dilation = 2 ** i
            in_ch = input_size if i == 0 else num_channels[i - 1]
            out_ch = num_channels[i]
            layers += [
                nn.Conv1d(in_ch, out_ch, kernel_size, padding=dilation, dilation=dilation),
                nn.ReLU(), nn.Dropout(dropout)
            ]
        self.network = nn.Sequential(*layers)

    def forward(self, x):
        return self.network(x)

class ClimateTCNModel(nn.Module):
    def __init__(self, input_channels, height, width, timesteps, embed_dim=64):
        super().__init__()
        self.encoder = ChEncoder(input_channels, embed_dim)
        self.tcn = TemporalConvNet(embed_dim, [128, 128])
        self.fc = nn.Sequential(
            nn.Linear(128, 256), nn.ReLU(),
            nn.Linear(256, height * width)
        )
        self.height = height
        self.width = width

    def forward(self, x):
        B, T, C, H, W = x.shape
        x = x.view(B * T, C, H, W)
        x = self.encoder(x)
        x = x.view(B, T, -1).transpose(1, 2)
        x = self.tcn(x)
        x = x[:, :, -1]
        out = self.fc(x)
        return out.view(B, self.height, self.width)

# ===== 读取数据 =====
ds = xr.open_dataset(data_path)

# 选取 2024-05 至 2025-07 共15个月
ds_sel = ds.sel(time=slice("2025-05-01", "2026-07-01"))

# 空间维度
H, W = ds_sel[variables[0]].shape[1:]

# 变量堆叠
data_stack = np.stack([ds_sel[var].values for var in variables], axis=1)  # (T, C, H, W)

# ===== 时间编码 =====
month = (ds_sel.time.dt.month.values - 1).astype(np.float32)
month_sin = np.sin(2 * np.pi * month / 12)
month_cos = np.cos(2 * np.pi * month / 12)
month_sin_2d = np.tile(month_sin[:, None, None, None], (1, 1, H, W))
month_cos_2d = np.tile(month_cos[:, None, None, None], (1, 1, H, W))
data_stack = np.concatenate([data_stack, month_sin_2d, month_cos_2d], axis=1)  # (T, 8, H, W)

# ===== 标准化 =====
mean = data_stack[:, :6].mean(axis=(0, 2, 3), keepdims=True)
std = data_stack[:, :6].std(axis=(0, 2, 3), keepdims=True)
std[std == 0] = 1.0
data_stack[:, :6] = (data_stack[:, :6] - mean) / std

# ===== 加载模型 =====
model = ClimateTCNModel(input_channels=8, height=H, width=W, timesteps=timesteps).to(device)
model.load_state_dict(torch.load(model_path, map_location=device))
model.eval()

# ===== 推理 2025-09 (只一次) =====
x_tensor = torch.tensor(data_stack, dtype=torch.float32).unsqueeze(0).to(device)
with torch.no_grad():
    pred = model(x_tensor).cpu().numpy()[0]  # (H, W)

# ===== 保存结果 =====
pred_da = xr.DataArray(
    pred,
    coords={"latitude": ds.latitude, "longitude": ds.longitude},
    dims=["latitude", "longitude"],
    name="predicted_tp"
)

# 添加时间坐标为 2025-09-01
pred_da = pred_da.expand_dims(time=[np.datetime64("2026-08-01")])

pred_da.to_netcdf(output_path)
print(f"✅ 推理完成，保存至 {output_path}")


In [ ]:
import xarray as xr
import matplotlib.pyplot as plt
import cartopy.crs as ccrs
import cartopy.feature as cfeature
import numpy as np

# ===== 文件路径 =====
file_path = "/Users/zhangnan/日常文件/ai/每周下载和预测/53 20260831/20260831-tp/20260831-tp-1/预测结果-tp-2026年8月.nc"

# 读取数据
ds = xr.open_dataset(file_path)
da = ds["predicted_tp"].squeeze()  # 去掉 time 维度
lons, lats = np.meshgrid(ds.longitude, ds.latitude)

# ===== 绘图 =====
plt.figure(figsize=(12, 6), dpi=300)
proj = ccrs.PlateCarree()
ax = plt.axes(projection=proj)

# 设置地图范围，可根据需要修改
ax.set_global()

# 添加地理要素
ax.add_feature(cfeature.COASTLINE, linewidth=0.8)
ax.add_feature(cfeature.BORDERS, linestyle=':', linewidth=0.6)
ax.add_feature(cfeature.LAND, facecolor='lightgray')
ax.add_feature(cfeature.OCEAN, facecolor='white')

# 绘制填色图
cmap = plt.cm.viridis
levels = np.linspace(float(da.min()), float(da.max()), 21)
cf = ax.contourf(lons, lats, da, levels=levels, cmap=cmap,
                 transform=ccrs.PlateCarree(), extend='both')

# 添加色标
cbar = plt.colorbar(cf, orientation='horizontal', pad=0.05, aspect=40, shrink=0.8)
cbar.set_label("Predicted Precipitation (mm/month)", fontsize=12)

# 添加网格线及标签
gl = ax.gridlines(draw_labels=True, linewidth=0.5, color='gray', alpha=0.5, linestyle='--')
gl.top_labels = False
gl.right_labels = False
gl.xlabel_style = {'size': 10}
gl.ylabel_style = {'size': 10}

# 添加标题
plt.title("Predicted Precipitation for August 2026",
          fontsize=14, weight='bold', pad=10)

# 保存或显示
plt.tight_layout()
plt.show()


In [ ]:
# 3 重采样tp era5-daily到周的1.5度
import os
import xarray as xr
import numpy as np
from tqdm import tqdm

# ===============================
# 输入输出路径
# ===============================
input_file = '/Users/zhangnan/日常文件/ai/每周下载和预测/53 20260831/数据/data/ERA5-daily-single level-Totalprecipiation-20260831.nc'
output_base_dir = '/Users/zhangnan/日常文件/ai/每周下载和预测/53 20260831/20260831-tp/20260831-tp-1/tp周'

# 目标插值经纬度
target_lat = np.linspace(90, -90, 121)
target_lon = np.linspace(0, 358.5, 240)

# ===============================
# 打开数据集
# ===============================
ds = xr.open_dataset(input_file)
tp = ds['tp']  # (valid_time, latitude, longitude)

# 单位转换：m -> mm
tp_mm = tp * 1000.0
tp_mm.name = 'tp'
tp_mm.attrs['units'] = 'mm'
tp_mm.attrs['long_name'] = 'Total precipitation'
tp_mm.attrs['standard_name'] = 'precipitation_amount'

# ===============================
# 周平均（周一为起始日）
# ===============================
tp_weekly = tp_mm.resample(valid_time='1W-MON').mean()

# 插值到目标网格
tp_interp = tp_weekly.interp(latitude=target_lat, longitude=target_lon, method='linear')

# ===============================
# 逐周保存
# ===============================
for i in tqdm(range(tp_interp.valid_time.size), desc='保存每周数据'):
    da = tp_interp.isel(valid_time=i)
    date_str = np.datetime_as_string(da.valid_time.values, unit='D').replace('-', '')
    year = date_str[:4]  # 从日期提取年份

    output_path = os.path.join(output_base_dir, f'obs-era5-{date_str}-tp.nc')
    os.makedirs(os.path.dirname(output_path), exist_ok=True)

    # 构造数据集并添加时间坐标
    da_ds = xr.Dataset({da.name: da})
    da_ds = da_ds.expand_dims('time')
    da_ds = da_ds.assign_coords(time=[da.valid_time.values])
    da_ds['time'].attrs = {
        'standard_name': 'time',
        'long_name': 'time',
        'bounds': 'time_bnds',
        'axis': 'T'
    }

    # 添加 variable 坐标
    da_ds['variable'] = xr.DataArray(['tp'], dims='variable')
    da_ds['variable'].attrs = {
        'long_name': 'variable name',
        'standard_name': 'variable',
    }

    # 保存为 NetCDF 文件
    da_ds.to_netcdf(output_path)


In [ ]:
# 4 处理到周的200、300、500gh
import os
import xarray as xr
import numpy as np
import pandas as pd

# === 输入路径和输出路径 ===
input_dir = "/Users/zhangnan/日常文件/ai/每周下载和预测/53 20260831/数据/data/"
output_path = "/Users/zhangnan/日常文件/ai/每周下载和预测/53 20260831/20260831-tp/20260831-tp-1/obs-era5-20260831-gh.nc"
os.makedirs(os.path.dirname(output_path), exist_ok=True)

# === 文件路径 ===
file_200 = os.path.join(input_dir, "ERA5-daily-200hpa-Geopotential-20260831.nc")
file_300 = os.path.join(input_dir, "ERA5-daily-300hpa-Geopotential-20260831.nc")
file_500 = os.path.join(input_dir, "ERA5-daily-500hpa-Geopotential-20260831.nc")

# === 插值目标网格 ===
target_lat = np.arange(90, -91, -1.5)
target_lon = np.arange(0, 360, 1.5)

# === 读取并转换为位势高度 ===
ds200 = xr.open_dataset(file_200)
ds300 = xr.open_dataset(file_300)
ds500 = xr.open_dataset(file_500)

z200 = ds200['z'].squeeze(dim='pressure_level') / 9.8
z300 = ds300['z'].squeeze(dim='pressure_level') / 9.8
z500 = ds500['z'].squeeze(dim='pressure_level') / 9.8

# === 插值 ===
z200_interp = z200.interp(latitude=target_lat, longitude=target_lon, method="linear")
z300_interp = z300.interp(latitude=target_lat, longitude=target_lon, method="linear")
z500_interp = z500.interp(latitude=target_lat, longitude=target_lon, method="linear")

# === 缺失值填充 ===
z200_filled = z200_interp.ffill('latitude').bfill('latitude').ffill('longitude').bfill('longitude')
z300_filled = z300_interp.ffill('latitude').bfill('latitude').ffill('longitude').bfill('longitude')
z500_filled = z500_interp.ffill('latitude').bfill('latitude').ffill('longitude').bfill('longitude')

# === 差值计算 ===
z200_z300 = z200_filled - z300_filled
z200_z500 = z200_filled - z500_filled

# === 周平均 ===
time_dim = 'valid_time'
z200_weekly = z200_filled.resample({time_dim: '1W-MON'}).mean()
z200_z300_weekly = z200_z300.resample({time_dim: '1W-MON'}).mean()
z200_z500_weekly = z200_z500.resample({time_dim: '1W-MON'}).mean()

# === 保存为单个 NetCDF 文件 ===
ds_out = xr.Dataset({
    'z200_weekly': z200_weekly,
    'z200_z300_weekly': z200_z300_weekly,
    'z200_z500_weekly': z200_z500_weekly
})

ds_out.to_netcdf(output_path)
print(f"✅ 成功保存至 {output_path}")
print(ds_out)


In [ ]:
# 5 处理到周的700hPa-SpecificHumidity
import os
import xarray as xr
import numpy as np

# 输入文件
input_file = "/Users/zhangnan/日常文件/ai/每周下载和预测/53 20260831/数据/data/ERA5-daily-700hPa-SpecificHumidity-20260831.nc"

# 输出文件
output_file = "/Users/zhangnan/日常文件/ai/每周下载和预测/53 20260831/20260831-tp/20260831-tp-1/obs-era5-20260831-q700.nc"

# 目标插值网格
target_lat = np.linspace(90, -90, 121)
target_lon = np.linspace(0, 358.5, 240)

# 打开数据集
ds = xr.open_dataset(input_file)

# 选择 700hPa 比湿
q700 = ds['q'].sel(pressure_level=700).squeeze()
q700.name = 'q700'
q700.attrs['units'] = 'kg kg-1'
q700.attrs['long_name'] = 'Specific humidity at 700hPa'
q700.attrs['standard_name'] = 'specific_humidity'

# 按周平均 (以周一为起点)
q700_weekly = q700.resample(valid_time='1W-MON').mean()

# 插值到目标网格
q700_interp = q700_weekly.interp(latitude=target_lat, longitude=target_lon, method='linear')

# 创建 Dataset
ds_out = xr.Dataset({'q700': q700_interp})

# 处理时间坐标
ds_out = ds_out.rename({'valid_time': 'time'})
ds_out['time'].attrs = {
    'standard_name': 'time',
    'long_name': 'time',
    'bounds': 'time_bnds',
    'axis': 'T'
}

# 添加 variable 坐标
ds_out['variable'] = xr.DataArray(['q700'], dims='variable')
ds_out['variable'].attrs = {
    'long_name': 'variable name',
    'standard_name': 'variable'
}

# 确保输出目录存在
os.makedirs(os.path.dirname(output_file), exist_ok=True)

# 保存到 NetCDF
ds_out.to_netcdf(output_file)

print(f"已保存到 {output_file}")


In [ ]:
# 6 处理到周的800hPa-CloudFraction
import xarray as xr
import numpy as np
import os

# === 输入文件与输出路径 ===
input_file = "/Users/zhangnan/日常文件/ai/每周下载和预测/53 20260831/数据/data/ERA5-daily-800hPa-CloudFraction-20260831.nc"
output_file = "/Users/zhangnan/日常文件/ai/每周下载和预测/53 20260831/20260831-tp/20260831-tp-1/obs-era5-20260831-cloudfraction800.nc"

# 目标经纬度
target_lat = np.linspace(90, -90, 121)
target_lon = np.linspace(0, 358.5, 240)

# 打开数据集，选择 800hPa 层
ds = xr.open_dataset(input_file)
cc = ds['cc'].sel(pressure_level=800)  # (valid_time, latitude, longitude)

# 设置变量属性
cc.name = 'cloudfraction800'
cc.attrs['units'] = ds['cc'].attrs.get('units', '1')
cc.attrs['long_name'] = 'Cloud Fraction at 800hPa'
cc.attrs['standard_name'] = 'cloud_area_fraction_in_atmosphere_layer'

# 按周重采样（以周一为起始），取均值
cc_weekly = cc.resample(valid_time='1W-MON').mean()

# 插值到目标网格
cc_interp = cc_weekly.interp(latitude=target_lat, longitude=target_lon, method='linear')

# 合并所有周的数据到一个 Dataset
cc_ds = cc_interp.to_dataset(name='cloudfraction800')

# 添加 variable 坐标
cc_ds['variable'] = xr.DataArray(['cloudfraction800'], dims='variable')
cc_ds['variable'].attrs = {
    'long_name': 'variable name',
    'standard_name': 'variable',
}

# 添加时间属性
cc_ds['valid_time'].attrs = {
    'standard_name': 'time',
    'long_name': 'time',
    'bounds': 'time_bnds',
    'axis': 'T'
}

# 保存为 NetCDF 文件
os.makedirs(os.path.dirname(output_file), exist_ok=True)
cc_ds.to_netcdf(output_file)

print(f"处理完成，文件已保存至: {output_file}")


In [ ]:
# 7 合并周数据
import os
import xarray as xr
from tqdm import tqdm

# 输入文件路径
base_dir = "/Users/zhangnan/日常文件/ai/每周下载和预测/53 20260831/20260831-tp/20260831-tp-1"
tp_dir = os.path.join(base_dir, "tp周")
gh_file = os.path.join(base_dir, "obs-era5-20260831-gh.nc")
q700_file = os.path.join(base_dir, "obs-era5-20260831-q700.nc")
cloud_file = os.path.join(base_dir, "obs-era5-20260831-cloudfraction800.nc")

# 输出文件
output_path = os.path.join(base_dir, "tp-weekmerged_20260831.nc")

def drop_pressure_level(ds):
    """安全移除 pressure_level"""
    if "pressure_level" in ds.variables:
        ds = ds.drop_vars("pressure_level")
    elif "pressure_level" in ds.coords:
        ds = ds.reset_coords("pressure_level", drop=True)
    return ds

def merge_weekly_files():
    print("开始处理 20260831 TP 周数据合并...")

    tp_files = sorted([f for f in os.listdir(tp_dir) if f.endswith(".nc")])
    merged_datasets = []

    # 打开整段文件
    ds_gh = drop_pressure_level(xr.open_dataset(gh_file))
    ds_q700 = drop_pressure_level(xr.open_dataset(q700_file))
    ds_cloud = drop_pressure_level(xr.open_dataset(cloud_file))

    for i, tp_file in enumerate(tqdm(tp_files, desc="合并进度")):
        try:
            ds_tp = drop_pressure_level(xr.open_dataset(os.path.join(tp_dir, tp_file)))

            # 时间
            time_val = ds_tp['time'].values

            # 提取变量并扩展维度
            tp_sel = ds_tp['tp'].isel(time=0).expand_dims({'time': time_val})
            z200_weekly = ds_gh['z200_weekly'].isel(valid_time=i).expand_dims({'time': time_val})
            z200_z300_weekly = ds_gh['z200_z300_weekly'].isel(valid_time=i).expand_dims({'time': time_val})
            z200_z500_weekly = ds_gh['z200_z500_weekly'].isel(valid_time=i).expand_dims({'time': time_val})
            q700 = ds_q700['q700'].isel(time=i).expand_dims({'time': time_val})
            cloudfraction800 = ds_cloud['cloudfraction800'].isel(valid_time=i).expand_dims({'time': time_val})

            # 构建数据集
            ds_merged = xr.Dataset(
                data_vars={
                    'tp': tp_sel,
                    'z200_weekly': z200_weekly,
                    'z200_z300_weekly': z200_z300_weekly,
                    'z200_z500_weekly': z200_z500_weekly,
                    'q700': q700,
                    'cloudfraction800': cloudfraction800,
                },
                coords={
                    'time': time_val,
                    'latitude': ds_tp['latitude'],
                    'longitude': ds_tp['longitude'],
                }
            )
            merged_datasets.append(ds_merged)
            ds_tp.close()

        except Exception as e:
            print(f"  ❌ Error on {tp_file}: {e}")

    if not merged_datasets:
        print("❌ 没有成功合并的数据集，请检查文件")
        return

    ds_all = xr.concat(merged_datasets, dim='time', compat='override', coords='minimal').sortby('time')

    print(f"✅ 保存合并文件到: {output_path}")
    ds_all.to_netcdf(output_path)
    ds_all.close()
    ds_gh.close()
    ds_q700.close()
    ds_cloud.close()

    print("🎯 数据合并完成")

merge_weekly_files()


In [ ]:
# 8 周数据添加基准日期
import xarray as xr
import pandas as pd
import numpy as np
import os

# 输入输出路径
input_file = "/Users/zhangnan/日常文件/ai/每周下载和预测/53 20260831/20260831-tp/20260831-tp-1/tp-weekmerged_20260831.nc"
output_dir = "/Users/zhangnan/日常文件/ai/每周下载和预测/53 20260831/20260831-tp/20260831-tp-1/"
output_file = os.path.join(output_dir, "20260831_tp_weekly_sample.nc")

# 加载数据集并确保时间唯一
ds = xr.open_dataset(input_file)
ds['time'] = pd.to_datetime(ds['time'].values)
_, unique_idx = np.unique(ds['time'], return_index=True)
ds = ds.isel(time=unique_idx)

# 设置目标周日期
target_date = pd.to_datetime("2026-08-31")

# 构造过去 20 周（第4周到第23周）和 10 周（第4周到第13周）的日期列表
past_20_weeks = [target_date - pd.Timedelta(weeks=i) for i in range(4, 24)]
past_10_weeks = [target_date - pd.Timedelta(weeks=i) for i in range(4, 14)]

# 选择数据，自动忽略不存在的时间
tp_hist_all = ds['tp'].sel(time=past_20_weeks)
z200_hist_all = ds['z200_weekly'].sel(time=past_10_weeks)
z200_z300_hist_all = ds['z200_z300_weekly'].sel(time=past_10_weeks)
z200_z500_hist_all = ds['z200_z500_weekly'].sel(time=past_10_weeks)
q700_hist_all = ds['q700'].sel(time=past_10_weeks)
cloudfraction800_hist_all = ds['cloudfraction800'].sel(time=past_10_weeks)

# 构建新的 Dataset
data_vars = {}
coords = {"time": [target_date], "latitude": ds.latitude, "longitude": ds.longitude}

# 历史20周 tp
for i in range(tp_hist_all.time.size):
    data_vars[f"tp_hist_{i}"] = (("latitude", "longitude"), tp_hist_all.isel(time=i).values)

# 历史10周 z200_weekly
for i in range(z200_hist_all.time.size):
    data_vars[f"z200_hist_{i}"] = (("latitude", "longitude"), z200_hist_all.isel(time=i).values)

# 历史10周 z200_z300_weekly
for i in range(z200_z300_hist_all.time.size):
    data_vars[f"z200_z300_hist_{i}"] = (("latitude", "longitude"), z200_z300_hist_all.isel(time=i).values)

# 历史10周 z200_z500_weekly
for i in range(z200_z500_hist_all.time.size):
    data_vars[f"z200_z500_hist_{i}"] = (("latitude", "longitude"), z200_z500_hist_all.isel(time=i).values)

# 历史10周 q700
for i in range(q700_hist_all.time.size):
    data_vars[f"q700_hist_{i}"] = (("latitude", "longitude"), q700_hist_all.isel(time=i).values)

# 历史10周 cloudfraction800
for i in range(cloudfraction800_hist_all.time.size):
    data_vars[f"cloudfraction800_hist_{i}"] = (("latitude", "longitude"), cloudfraction800_hist_all.isel(time=i).values)

ds_sample = xr.Dataset(data_vars=data_vars, coords=coords)

# 保存到 NetCDF
os.makedirs(output_dir, exist_ok=True)
ds_sample.to_netcdf(output_file)
print(f"✅ 数据整理完成并保存：{output_file}")


In [ ]:
# 9 添加月预测数据
import os
import xarray as xr
import numpy as np

# === 输入文件路径 ===
weekly_file = "/Users/zhangnan/日常文件/ai/每周下载和预测/53 20260831/20260831-tp/20260831-tp-1/20260831_tp_weekly_sample.nc"
pred_file = "/Users/zhangnan/日常文件/ai/每周下载和预测/53 20260831/20260831-tp/20260831-tp-1/预测结果-tp-2026年8月.nc"

# === 输出文件路径 ===
output_file = "/Users/zhangnan/日常文件/ai/每周下载和预测/53 20260831/20260831-tp/20260831-tp-1/tp_weekly_with_monthly_pred.nc"
os.makedirs(os.path.dirname(output_file), exist_ok=True)

# === 打开数据集 ===
ds_weekly = xr.open_dataset(weekly_file)
ds_pred = xr.open_dataset(pred_file)

# 确保经度为 [0, 360] 并排序
ds_pred = ds_pred.assign_coords(longitude=(ds_pred.longitude % 360)).sortby("longitude")

# 取预测变量
pred_tp = ds_pred["predicted_tp"]

# 获取周数据的时间并对应到月份
time_week = ds_weekly["time"].values[0]
month_start = np.datetime64(f"{time_week.astype('datetime64[M]')}")  # 月首

# 检查预测数据是否包含该月
if month_start not in pred_tp.time:
    print(f"❌ 缺失该月份预测数据: {month_start}")
    shape = (len(ds_weekly.latitude), len(ds_weekly.longitude))
    pred_tp_values = np.full(shape, np.nan)
else:
    pred_tp_values = pred_tp.sel(time=month_start).values

# 构建 DataArray
pred_tp_da = xr.DataArray(
    data=pred_tp_values[np.newaxis, :, :],  # 增加时间维度
    dims=("time", "latitude", "longitude"),
    coords={
        "time": ds_weekly.time,
        "latitude": ds_weekly.latitude,
        "longitude": ds_weekly.longitude
    },
    name="pred_tp_month",
    attrs={"description": "对应周的月份预测降水量"}
)

# 添加到周数据集
ds_weekly["pred_tp_month"] = pred_tp_da

# 保存
ds_weekly.to_netcdf(output_file)
print(f"💾 已保存: {output_file}")


In [ ]:
# 10 最后一步，添加地形数据
import xarray as xr
import numpy as np
import os

# 文件路径
input_file = "/Users/zhangnan/日常文件/ai/每周下载和预测/53 20260831/20260831-tp/20260831-tp-1/tp_weekly_with_monthly_pred.nc"
output_file = "/Users/zhangnan/日常文件/ai/每周下载和预测/53 20260831/20260831-tp/20260831-tp-1/tp_weekly_with_monthly_pred_with_elev.nc"

# 地形数据
elev_file = "/Users/zhangnan/日常文件/ai/每周下载和预测/推理用/地形数据/etopo_1.5deg.nc"
ds_elev = xr.open_dataset(elev_file)

# 将地形经度转换为 0~360 范围
def lon_180_to_360(lon):
    lon_360 = lon.copy()
    lon_360 = np.where(lon_360 < 0, lon_360 + 360, lon_360)
    return lon_360

ds_elev = ds_elev.assign_coords(lon=lon_180_to_360(ds_elev.lon))
ds_elev = ds_elev.sortby('lon')

# 打开目标文件
ds = xr.open_dataset(input_file)

# 重命名地形坐标与目标一致
ds_elev_renamed = ds_elev.rename({'lat': 'latitude', 'lon': 'longitude'})

# 插值到目标经纬度
ds_elev_interp = ds_elev_renamed.interp(
    latitude=ds.latitude,
    longitude=ds.longitude,
    method="linear"
)

# 扩展到时间维度
elev_expanded = ds_elev_interp['elevation'].expand_dims({'time': ds.time}, axis=0)
elev_expanded = elev_expanded.transpose('time', 'latitude', 'longitude')

# 新建 DataArray
elev_da = xr.DataArray(
    data=elev_expanded.values,
    dims=['time', 'latitude', 'longitude'],
    coords={'time': ds.time, 'latitude': ds.latitude, 'longitude': ds.longitude},
    attrs={
        'long_name': 'surface_elevation',
        'units': 'meters',
        'description': 'Surface elevation interpolated from etopo_1.5deg.nc and expanded along time dimension'
    }
)

# 添加到数据集
ds = ds.assign(elevation=elev_da)

# 保存新文件
ds.to_netcdf(output_file)
ds.close()

print(f"完成！新文件已保存: {output_file}")


In [ ]:

# 读取nc文件信息
import xarray as xr
# 定义文件路径
file_path = '/Users/zhangnan/日常文件/ai/每周下载和预测/53 20260831/20260831-tp/20260831-tp-1/tp_weekly_with_monthly_pred_with_elev.nc'
# 打印原文件路径
print(f"文件路径: {file_path}")
# 打开NetCDF文件
ds = xr.open_dataset(file_path)
# 输出数据集的基本信息
print(ds)
# 如果需要查看数据集的变量列表，可以使用
print(ds.variables)
# 如果需要查看数据集的维度，可以使用
#print(ds.dims)